# Contrastive Pair Export & Pilot Validation

This notebook joins the prediction log with the cached BridgeData V2 manifest,
runs the pilot validation of the directional-consistency metric against
early-motion ground truth, and exports `pairs.json` for the
external Isaac Sim visualiser.

The model is never loaded here; inputs are the CSVs produced by the previous
notebooks.

**Prerequisite:** predictions must have been logged with `pair_id`,
`role` (`'a'`/`'b'`) and `scene_id` passed via `**extra` in
`append_prediction_log`, with `scene_id` matching `episode_index` in the
manifest.


## 1. Mount Drive


In [29]:
from google.colab import drive
drive.mount('/content/drive')
import os
PRED_CSV = '/content/drive/MyDrive/openvla_cache/probe_predictions_v2.csv'
CACHE_DIR = '/content/drive/MyDrive/openvla_cache/bridge_multiobj'
OUT_JSON  = '/content/drive/MyDrive/openvla_cache/pairs.json'
MANIFEST_CSV = os.path.join(CACHE_DIR, 'manifest.csv')
print('predictions ->', PRED_CSV)
print('manifest    ->', MANIFEST_CSV)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
predictions -> /content/drive/MyDrive/openvla_cache/probe_predictions_v2.csv
manifest    -> /content/drive/MyDrive/openvla_cache/bridge_multiobj/manifest.csv


## 2. Import `export_pairs.py`

Adds the local repository directory to the import path and clears the import
cache, mirroring the pattern used for `model.py` and `data.py`.

On Colab the kernel working directory is usually `/content`, not the folder that
holds the notebook, so auto-detection also searches a mounted Drive. If that
fails, set `REPO_DIR` in the next cell to the folder that contains
`export_pairs.py`.


In [30]:
import sys, os, glob, importlib

# Colab cwd is usually /content, not the notebook folder. Set REPO_DIR to the
# folder that contains export_pairs.py when auto-detection fails, for example:
#   REPO_DIR = '/content/drive/MyDrive/ECS8056'
REPO_DIR = ''


def find_repo_dir(anchor):
    """Return the directory holding `anchor`, searching several sensible roots."""
    starts = [REPO_DIR, os.getcwd()]
    nb_path = globals().get('__vsc_ipynb_file__')
    if nb_path:
        starts.insert(0, os.path.dirname(os.path.abspath(nb_path)))
    try:
        starts += [str(p) for p in (get_ipython().user_ns.get('_dh') or [])]
    except Exception:
        pass
    for known in ('/content/ECS8056', '/content/drive/MyDrive/ECS8056'):
        starts.append(known)

    seen = set()
    for start in starts:
        if not start:
            continue
        d = os.path.abspath(start)
        for _ in range(6):
            if d in seen:
                break
            seen.add(d)
            if os.path.isfile(os.path.join(d, anchor)):
                return d
            parent = os.path.dirname(d)
            if parent == d:
                break
            d = parent

    drive_root = '/content/drive/MyDrive'
    if os.path.isdir(drive_root):
        for depth in range(6):
            hits = glob.glob(os.path.join(drive_root, *(['*'] * depth), anchor))
            if hits:
                return os.path.dirname(os.path.abspath(hits[0]))
    return None


module_dir = find_repo_dir('export_pairs.py')
if module_dir is None:
    raise FileNotFoundError(
        "export_pairs.py not found. On Colab, open or upload the whole "
        "repository (not only the notebook), mount Drive, then set REPO_DIR "
        f"above to the folder that contains export_pairs.py. cwd={os.getcwd()!r}.")
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)
sys.modules.pop('export_pairs', None)
importlib.invalidate_caches()

import export_pairs
from export_pairs import load_inputs, build_pairs, write_pairs
print('imported export_pairs.py from', module_dir)
print('BRIDGE_TO_ISAAC =\n', export_pairs.BRIDGE_TO_ISAAC)

imported export_pairs.py from /content/ECS8056
BRIDGE_TO_ISAAC =
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


## 3. Load the prediction log and manifest

`load_inputs` validates that the prediction log carries the required probe
columns and fails loudly if any are missing.


In [31]:
preds, manifest = load_inputs(PRED_CSV, MANIFEST_CSV)
print(f'{len(preds)} logged predictions | {preds["pair_id"].nunique()} pair ids | '
      f'{preds["scene_id"].nunique()} scenes')
print(f'{len(manifest)} manifest rows')
preds.head(4)

144 logged predictions | 72 pair ids | 72 scenes
224 manifest rows


,timestamp,instruction,unnorm_key,do_sample,gpu_name,gpu_capability,dtype,seed,torch,transformers,...,a4,a5,a6,scene_id,pair_id,role,spatial_term,category,feasible_both,sample_idx
0,2026-07-26T13:23:22.837563+00:00,Place the can to the left of the pot.,bridge_orig,False,NVIDIA A100-SXM4-80GB,sm_80,bfloat16,42,2.11.0+cu128,4.40.1,...,0.025344,0.007992,0.996078,0,ep000000_left,a,left,placement_relation,yes,0
1,2026-07-26T13:23:23.444073+00:00,Place the can to the right of the pot.,bridge_orig,False,NVIDIA A100-SXM4-80GB,sm_80,bfloat16,42,2.11.0+cu128,4.40.1,...,0.030033,-0.009737,0.996078,0,ep000000_left,b,left,placement_relation,yes,0
2,2026-07-26T13:23:24.078268+00:00,Slide the cloth diagonally to the front of the...,bridge_orig,False,NVIDIA A100-SXM4-80GB,sm_80,bfloat16,42,2.11.0+cu128,4.40.1,...,0.022665,-0.008126,0.996078,2,ep000002_front,a,front,referent_selection,no,0
3,2026-07-26T13:23:24.683984+00:00,Slide the cloth diagonally to the back of the ...,bridge_orig,False,NVIDIA A100-SXM4-80GB,sm_80,bfloat16,42,2.11.0+cu128,4.40.1,...,0.024674,0.001545,0.996078,2,ep000002_front,b,front,referent_selection,no,0


## 4. Build contrastive pairs

Predictions are grouped by `pair_id`; repeated predictions per role
(samples or paraphrases) are averaged, retaining per-axis standard deviation
and count for the effect-size analysis. Pair ids lacking both roles are
reported and skipped.


In [32]:
pairs, skipped = build_pairs(preds, manifest)
with_gt = sum(1 for p in pairs if 'gt_vector' in p)
print(f'{len(pairs)} complete pairs | {skipped} incomplete pair ids skipped | '
      f'{with_gt} pairs joined to ground truth')

72 complete pairs | 0 incomplete pair ids skipped | 72 pairs joined to ground truth


## 5. Pilot validation - sign agreement against ground truth

Gate for the directional metric and the frame convention: the metrics are
validated against trajectories whose ground-truth motion direction is known.
For each prediction, the sign of the predicted translation on the dominant
ground-truth axis is compared with the ground-truth sign.

Interpretation:
* **Near-zero agreement on one axis** indicates a flipped axis between the
  Bridge action frame and the assumed frame, set the corresponding row of
  `BRIDGE_TO_ISAAC` in `export_pairs.py` and rebuild.
* **Near-chance agreement on all axes** means the check carries no signal on
  this subset; restrict to scenes with large, unambiguous ground-truth motion
  before concluding anything about the frame.
* Model failure is expected on *some* scenes so the gate is systematic, 
  axis-level disagreement, not per-scene misses.


In [33]:
import numpy as np
import pandas as pd

MIN_GT_NORM = 0.01   # exclude near-static ground truth; tune against the data

rows = []
for p in pairs:
    gt = p.get('gt_vector')
    if gt is None:
        continue
    gt_t = np.asarray(gt[:3])
    if np.linalg.norm(gt_t) < MIN_GT_NORM:
        continue
    dom = int(np.argmax(np.abs(gt_t)))
    for role in ('a', 'b'):
        act_t = np.asarray(p[f'action_{role}'][:3])
        rows.append({
            'pair_id': p['pair_id'],
            'role': role,
            'dominant_axis': 'xyz'[dom],
            'gt_sign': float(np.sign(gt_t[dom])),
            'pred_sign': float(np.sign(act_t[dom])),
            'agree': bool(np.sign(act_t[dom]) == np.sign(gt_t[dom])),
        })

pilot = pd.DataFrame(rows)
print(pilot.groupby('dominant_axis')['agree'].agg(['mean', 'count']))
print(f"\noverall sign agreement: {pilot['agree'].mean():.1%} "
      f"over {len(pilot)} predictions ({pilot['pair_id'].nunique()} pairs)")

                   mean  count
dominant_axis                 
x              0.685185     54
y              0.700000     80
z              0.800000     10

overall sign agreement: 70.1% over 144 predictions (72 pairs)


## 5b. Stratified analysis - primary vs. secondary

The evaluation is split to avoid two confounds:

* **Referent vs. placement.** Only `referent_selection` prompts (the spatial
  term selects which object to grasp) probe grounding at the first control step.
  `placement_relation` prompts differ only in a destination, and the first
  action is the reach, identical for both variants, so they are reported
  separately and never pooled into the headline result.
* **Feasibility.** Antonym-swapped prompts can imply physically impossible
  placements; the primary stratum keeps only scenes reviewed as feasible on
  **both** sides (`feasible_both == 'yes'`).

**Headline statistic:** the paired difference `action_A − action_B` on the
lateral axis (`dx`, the axis expected to flip for left/right), tested with a
Wilcoxon signed-rank test across pairs. The per-pair sign-flip rate is retained
as a descriptive statistic only.

In [34]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

LATERAL_AXIS = 0   # dx: the translation component expected to flip for left/right

# One row per pair, with the lateral-axis paired difference and its sign flip.
rec = []
for p in pairs:
    a = float(p['action_a'][LATERAL_AXIS])
    b = float(p['action_b'][LATERAL_AXIS])
    rec.append({
        'pair_id': p['pair_id'],
        'category': p.get('category', 'other'),
        'feasible_both': p.get('feasible_both', 'unreviewed'),
        'spatial_term': p.get('spatial_term', ''),
        'lat_a': a,
        'lat_b': b,
        'diff': a - b,               # action_A - action_B on the lateral axis
        'sign_flip': (a * b) < 0,
    })
df = pd.DataFrame(rec)


def summarise(sub: pd.DataFrame, label: str):
    """Print counts, sign-flip rate (descriptive) and Wilcoxon on the diffs."""
    n = len(sub)
    print(f'\n[{label}] n={n} pairs')
    if n == 0:
        return
    print(f'  sign-flip rate (descriptive): {sub["sign_flip"].mean():.1%}')
    diffs = sub['diff'].to_numpy()
    if n >= 1 and np.any(diffs != 0):
        try:
            stat, pval = wilcoxon(diffs)
            print(f'  Wilcoxon signed-rank (A-B, lateral): '
                  f'W={stat:.1f}, p={pval:.4g}, median diff={np.median(diffs):+.4f}')
        except ValueError as e:
            print(f'  Wilcoxon not computable: {e}')
    else:
        print('  Wilcoxon not computable: all paired differences are zero')


# --- PRIMARY: referent_selection, feasible on both sides ---
primary = df[(df['category'] == 'referent_selection') &
             (df['feasible_both'] == 'yes')]
print('=' * 60)
print('PRIMARY ANALYSIS')
summarise(primary, 'referent_selection & feasible_both==yes')

# --- SECONDARY strata: reported separately, never pooled into the primary ---
print('\n' + '=' * 60)
print('SECONDARY STRATA (reported separately, not pooled)')
summarise(df[df['category'] == 'placement_relation'], 'placement_relation (all)')
summarise(df[(df['category'] == 'referent_selection') &
             (df['feasible_both'] != 'yes')],
          'referent_selection & infeasible/unreviewed')
summarise(df[df['category'] == 'other'], 'other')

print('\nstratum sizes:')
print(df.groupby(['category', 'feasible_both']).size())

PRIMARY ANALYSIS

[referent_selection & feasible_both==yes] n=4 pairs
  sign-flip rate (descriptive): 0.0%
  Wilcoxon signed-rank (A-B, lateral): W=0.0, p=0.5, median diff=-0.0009

SECONDARY STRATA (reported separately, not pooled)

[placement_relation (all)] n=64 pairs
  sign-flip rate (descriptive): 12.5%
  Wilcoxon signed-rank (A-B, lateral): W=264.5, p=0.2818, median diff=+0.0000

[referent_selection & infeasible/unreviewed] n=4 pairs
  sign-flip rate (descriptive): 25.0%
  Wilcoxon signed-rank (A-B, lateral): W=4.0, p=0.875, median diff=+0.0003

[other] n=0 pairs

stratum sizes:
category            feasible_both
placement_relation  no               18
                    unreviewed        2
                    yes              44
referent_selection  no                4
                    yes               4
dtype: int64


## 6. Export `pairs.json`

Written to Drive, then transferred to the rendering instance, e.g.:

```
scp -i key.pem pairs.json ubuntu@<instance-ip>:~/
```

The visualiser is run on the instance with Isaac Sim's bundled interpreter:
`./python.sh visualise_pairs.py --pairs ~/pairs.json --out ./figs`.


In [35]:
write_pairs(pairs, OUT_JSON)

[write_pairs] wrote 72 pairs -> /content/drive/MyDrive/openvla_cache/pairs.json


'/content/drive/MyDrive/openvla_cache/pairs.json'

## 7. Preview a pair

Spot check of one exported record: instructions, mean action vectors, and the
x-axis sign relationship that the left/right probes target.


In [36]:
import numpy as np
p = pairs[0]
print('pair_id :', p['pair_id'], '| scene:', p['scene_id'])
print('A:', p['instr_a'])
print('   action =', np.round(p['action_a'], 4), f"(n={p['n_a']})")
print('B:', p['instr_b'])
print('   action =', np.round(p['action_b'], 4), f"(n={p['n_b']})")
dx_a, dx_b = p['action_a'][0], p['action_b'][0]
print(f'dx(A) = {dx_a:+.4f}   dx(B) = {dx_b:+.4f}   '
      f"x-sign flip: {'YES' if dx_a * dx_b < 0 else 'no'}")
if 'gt_vector' in p:
    print('gt      =', np.round(p['gt_vector'], 4))

pair_id : ep000000_left | scene: 0
A: Place the can to the left of the pot.
   action = [-2.700e-03  5.000e-04 -8.000e-03  1.100e-03  2.530e-02  8.000e-03
  9.961e-01] (n=1)
B: Place the can to the right of the pot.
   action = [-2.700e-03  5.000e-04 -8.800e-03  1.100e-03  3.000e-02 -9.700e-03
  9.961e-01] (n=1)
dx(A) = -0.0027   dx(B) = -0.0027   x-sign flip: no
gt      = [-0.0868  0.0915 -0.0258 -0.0594 -0.035   1.4381  1.    ]
